In [0]:
# Databricks notebook source
import json
from datetime import datetime

dbutils.widgets.text("run_id", "")
dbutils.widgets.text("table_metadata", "")

run_id = dbutils.widgets.get("run_id")
table_metadata = dbutils.widgets.get("table_metadata")

if not run_id or not table_metadata:
    raise ValueError("run_id and table_metadata are required")

table_metadata = json.loads(table_metadata.replace("'", '"'))

#table_id = int(table_metadata["table_id"])
#table_name = table_metadata["table_name"]

start_time = datetime.utcnow()

print("Run ID:", run_id)
#print("Table ID:", table_id)
#print("Table Name:", table_name)

In [0]:
# =====================================================
# 2️⃣ Extract Variables
# =====================================================

table_id = int(table_metadata["table_id"])
table_name = table_metadata["table_name"]
source_system = table_metadata["source_system"].lower()
source_schema = table_metadata["source_schema"]
source_table = table_metadata["source_table"]
source_path = table_metadata["source_path"]
bronze_schema = table_metadata["bronze_schema"]
target_schema = table_metadata["target_layer"]

#load_type = table_parameters.get("load_type")
#watermark_column = table_parameters.get("watermark_column")

source_for_dim= f"wx_dbxproject.{target_schema}.{source_table}"
targetdim_table_fqn = f"wx_dbxproject.{target_schema}.{table_name}"

print(f"Source Table Name: {source_for_dim}")
print(f"Target Dimension Table: {targetdim_table_fqn}")

In [0]:
entry_exists = spark.sql(f"""
    SELECT 1
    FROM wx_dbxproject.metadata.pipeline_runs
    WHERE run_id = {run_id}
    AND table_id = {table_id}
    LIMIT 1
""").count() > 0

if entry_exists:
    
    spark.sql(f"""
        UPDATE wx_dbxproject.metadata.pipeline_runs
        SET
            layer = '02_silver',
            start_time = TIMESTAMP('{start_time}'),
            end_time = NULL,
            status = 'INPROGRESS',
            number_of_records = NULL,
            error_message = NULL
        WHERE run_id = {run_id}
        AND table_id = {table_id}
    """)

else:

    spark.sql(f"""
        INSERT INTO wx_dbxproject.metadata.pipeline_runs
        VALUES (
            {run_id},
            {table_id},
            '02_silver',
            TIMESTAMP('{start_time}'),
            NULL,
            'INPROGRESS',
            NULL,
            NULL
        )
    """)

print("Audit entry created / updated")

In [0]:
# =====================================================
# 4️⃣ Read Source
# =====================================================
status = "SUCCESS"
error_message = None
records_read = 0
records_written = 0

try:
    if source_system == "wx_dbxproject":
        source_df = spark.read.table(source_for_dim)
        source_df.write.format("delta").mode("overwrite").saveAsTable(targetdim_table_fqn)
    else:
        raise ValueError("Unsupported source_system")

    records_read = source_df.count()

    print("Dimension Creation Completed Successfully.")
    # print("Watermark will be updated after Silver load.")
    status='completed'

except Exception as e:
    end_time = spark.sql("SELECT current_timestamp()").collect()[0][0]
    error_message = str(e)

    spark.sql(f"""
        UPDATE wx_dbxproject.metadata.pipeline_runs
        SET
            end_time = TIMESTAMP('{end_time}'),
            status = 'FAILED',
            error_message = {'NULL' if not error_message else "'" + error_message.replace("'", "") + "'"}
        WHERE table_id = {table_id} AND run_id = {run_id} 
    """)
    raise

In [0]:
end_time = datetime.utcnow()

spark.sql(f"""
    UPDATE wx_dbxproject.metadata.pipeline_runs
    SET
        end_time = TIMESTAMP('{end_time}'),
        status = '{status}',
        error_message = {f"'{error_message}'" if error_message else 'NULL'}
    WHERE run_id = {run_id}
    AND table_id = {table_id}
""")

print("Audit table updated")